In [1]:
import faiss
import pickle

from sentence_transformers import SentenceTransformer

In [2]:
index = faiss.read_index(
    "vectorstore/oct_index.faiss"
)

with open(
    "vectorstore/metadata.pkl",
    "rb"
) as f:

    data = pickle.load(f)

chunks = data["chunks"]
metadata = data["metadata"]

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
def retrieve(query, k=10):

    q = embedder.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(q, k)

    results = []

    for idx in indices[0]:

        results.append({
            "text": chunks[idx],
            "meta": metadata[idx]
        })

    return results

In [4]:
results = retrieve(
    "RNFL thinning",
    k=10
)

for r in results:
    print(r["meta"])

{'filename': 'C034.rnfl.pdf', 'label': 'Healthy'}
{'filename': 'C013.rnfl.pdf', 'label': 'Healthy'}
{'filename': 'C031.rnfl.pdf', 'label': 'Healthy'}
{'filename': 'C028.rnfl.pdf', 'label': 'Healthy'}
{'filename': 'C005.pdf', 'label': 'Healthy'}
{'filename': 'C014.rnfl.pdf', 'label': 'Healthy'}
{'filename': 'C009.pdf', 'label': 'Healthy'}
{'filename': 'C044.rnfl.pdf', 'label': 'Healthy'}
{'filename': 'C041.rnfl.pdf', 'label': 'Healthy'}
{'filename': 'C024.rnfl.pdf', 'label': 'Healthy'}


In [5]:
def retrieve_both_groups():

    healthy = []

    schizophrenia = []

    for i in range(len(chunks)):

        if metadata[i]["label"] == "Healthy":
            healthy.append(chunks[i])

        elif metadata[i]["label"] == "Schizophrenia":
            schizophrenia.append(chunks[i])

    return healthy[:10] + schizophrenia[:10]

In [6]:
docs = retrieve_both_groups()

context = "\n\n".join(docs)

In [7]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:4b"
)

In [26]:
query = """
From OCT report C001.pdf, RNFL single exam report OU, give me the values of RNFL thickness at S, I, N, T.  
"""

In [27]:
prompt = f"""
Answer ONLY using the provided context.

Context:
{context}

Question:
{query}
"""

In [28]:
response = llm.invoke(prompt)

print(response.content)

Based on the provided information and standard ophthalmic terminology for Heidelberg Retina Tomography (HRF) reports, the RNFL (Retinal Nerve Fiber Layer) thickness values are typically reported in **four specific sectors**. Here's the precise breakdown for the query:

### Key Clarification:
- The report **does not** have a single "Superior" (S) or "Inferior" (I) value in isolation. Instead, Heidelberg HRF reports use **sector-specific labels**:
  - **TMP**: Temporal (outer side)
  - **TS**: Temporal Superior (part of the superior sector)
  - **NS**: Nasal Superior (part of the superior sector)
  - **NI**: Nasal Inferior (part of the inferior sector)
  - **INF**: Inferior (full sector, but this value is not explicitly listed in the provided report)
- In clinical practice, **"Superior" (S)** is often the **average of TS and NS**, and **"Inferior" (I)** is the **average of NI and INF**. However, the report here only provides four discrete values (TMP, TS, NS, NI), with no explicit "INF" 